# 🧮 01 · Python & NumPy for biologists: images, volumes and density maps
### A cryo-EM map is "just" a 3D grid of numbers — learn to slice it, project it, measure it, open its file and build one from atoms

**BiologyToResearchV3** · Part A — Foundations · Module 01 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/01_python_arrays_density_maps.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`ndimage`), Matplotlib, scikit-image (one isosurface plot) |
| **Prerequisites** | [Module 00](00_welcome_cells_to_structures.ipynb) (the 10-minute Python crash course). No calculus or physics. |
| **Estimated time** | 90–120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Zhang, Vasishtan, Xu, Topf & Alber 2010 (fitting atomic structures into density maps — preview); Xu, Zhang & Alber 2009 (rotation-invariant features of density maps — preview) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments in this course are small, controlled *toy versions* of published methods. They teach the core idea
> of each paper; they do **not** reproduce the papers' numbers. The density maps come from real Protein Data Bank (PDB)
> entries (via the open-source PolNet simulator) and a real cryo-EM map of β-galactosidase (NGL viewer test data); if
> you are offline, the notebook falls back to simple stand-in shapes and says so.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Create, index and slice** NumPy arrays, and explain the course convention `vol[z, y, x]` (counting from 0).
2. **Use** vectorised arithmetic, reductions (`mean`, `sum(axis=…)`), boolean masks and broadcasting instead of slow
   Python loops — and **avoid** two classic traps (views vs copies, repeated fancy indices).
3. **Display** 2D images with sensible colormaps and contrast, and **show** a 3D map as orthogonal slices, projections
   and an isosurface.
4. **Convert** between voxels, ångströms and nanometres, and **measure** what binning does to a real cryo-EM map.
5. **Read** an MRC density-map file byte by byte (header words → array) and **write** your own MRC file.
6. **Build** a density map from the atoms of a PDB file, **estimate** the molecule's mass, and **compare** your map with
   a reference map by Pearson correlation.
7. **Make** a computation reproducible with random seeds and saved outputs.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, import tools, load the course toolkit | 5 min |
| 1 · Arrays | lists vs arrays, shape/dtype, indexing, slicing, masks, broadcasting | 25 min |
| 2 · 2D images | colormaps, contrast, histograms, a noisy "micrograph" | 10 min |
| 3 · 3D volumes | orthogonal slices, projections, thresholds, an isosurface | 20 min |
| 4 · Units and binning | Å ↔ nm ↔ voxels; bin a real β-galactosidase map | 10 min |
| 5 · The MRC file format | open a map file byte by byte; write your own | 15 min |
| 6 · From atoms to density | parse a PDB file, estimate mass, build a map, compare with PolNet | 20 min |
| 7 · Reproducibility | seeds, saved figures, saved arrays | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

When a structural biologist says "we solved the structure of the ribosome at 3 Å", what do they actually *have* on their
hard drive? Not a picture, not a cartoon of ribbons — they have a **density map**: a 3D grid of numbers, each number
saying how much matter (strictly, how much electrostatic potential) sits in one tiny cube of space. A cellular tomogram
is the same kind of object, only bigger and much noisier. Every algorithm in this course — finding particles, aligning
them, averaging them, classifying them, segmenting membranes — is a recipe that **reads and rewrites grids of numbers**.

So before we can do any computational biology, we need fluency with these grids: how to look at them, cut them, measure
them in real units, store them in files, and connect them back to atoms. That is this module.

---
## 0 · Environment setup

Run the cells **in order, from the top** (click a cell, then **Shift + Enter**). The first cell checks which computer
you are on; a CPU is all we need.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("skimage", "scikit-image")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy import ndimage

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| SciPy", scipy.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the course's shared helper functions (from [`course_toolkit.py`](../course_toolkit.py)) that this
notebook uses: downloading maps (`fetch_zoo_map`, `fetch_ngl_map`), reading MRC files (`read_mrc` — we will take it
apart in Section 5), centring maps (`prepare_map`), correlation (`pearson`) and plotting (`show_orthoslices`).
Just run it; you do not need to read it now.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import struct
import urllib.request

POLNET_COMMIT = "73e613cc9162d26c7fd740ce3299bcb280a3e3b6"
POLNET_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
              + POLNET_COMMIT + "/data/templates/{kind}/{name}.{ext}")
NGL_COMMIT = "0db88033650614103be761f151efc923b66e977e"
NGL_URL = "https://raw.githubusercontent.com/nglviewer/ngl/" + NGL_COMMIT + "/data/{name}"
CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2277, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=5851, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3325, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=659, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=839, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1309, symmetry="C1"),
    # NB: PolNet's 4V94 map holds TWO TRiC complexes side by side (1.95 MDa ≈ 2 × ~1 MDa).
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1952, symmetry="C2"),
    "1bxn": dict(name="RuBisCO", organism="Cupriavidus necator", mass_kDa=280, symmetry="D4"),
    "1qvr": dict(name="ClpB chaperone", organism="Thermus thermophilus", mass_kDa=297, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=None, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=None, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=239, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=236, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=43, symmetry="C1"),
}


def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def procedural_molecule(pdb_id: str, size: int = 40) -> np.ndarray:
    """Offline fallback: a coarse shape with the right size/symmetry flavour (1 nm voxels)."""
    g = (np.arange(size) - size / 2 + 0.5)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2)

    def blob(cz, cy, cx, s):
        return np.exp(-((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * s**2))
    pid = pdb_id.lower()
    if pid in ("4v4r", "4v7r", "5mrc"):          # two unequal subunits
        v = blob(-3, 0, 0, 7.5) + 0.8 * blob(6, 2, 1, 5.5)
    elif pid in ("3j9i", "6utj", "4cr2"):        # stacked rings / barrel
        v = np.exp(-((r - 5.5)**2) / 4.0) * (np.abs(z) < 7.5)
        if pid != "3j9i":
            v = v + 0.8 * blob(10, 0, 0, 4)
    elif pid == "4v94":                          # double ring with cavity
        v = np.exp(-((r - 7)**2) / 6.0) * (np.abs(z) < 8)
    elif pid in ("3cf3", "1qvr"):                # hexameric ring
        ang = np.arctan2(y, x)
        v = np.exp(-((r - 5)**2) / 3.0) * (np.abs(z) < 4) * (1 + 0.5 * np.cos(6 * ang))
    elif pid == "1bxn":                          # compact ball
        v = blob(0, 0, 0, 4.5)
    else:                                        # small asymmetric blob
        v = blob(0, 0, 0, 3) + 0.5 * blob(2, 2, 0, 2)
    return ndimage.gaussian_filter(v.astype(np.float32), 1.0)


def fetch_zoo_map(pdb_id: str, voxel: str = "10A") -> tuple[np.ndarray, float, str]:
    """Return (density[z, y, x], voxel_size_Å, source) for a molecular-zoo entry.

    Tries the pinned PolNet template first; if the download fails (no internet),
    falls back to a *procedural* stand-in so the notebook still runs.
    """
    pdb_id = pdb_id.lower()
    dest = os.path.join(CACHE_DIR, f"polnet_{voxel}_{pdb_id}.mrc")
    url = POLNET_URL.format(kind=f"mrcs_{voxel}", name=pdb_id, ext="mrc")
    if _download(url, dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"PolNet template from PDB {pdb_id.upper()}"
    vol = procedural_molecule(pdb_id)
    return vol, 10.0, f"procedural stand-in for {pdb_id.upper()} (offline fallback)"


def fetch_ngl_map(name: str = "betaGal.mrc") -> tuple[np.ndarray, float, str] | None:
    """Download a real single-particle cryo-EM map distributed with the NGL viewer."""
    dest = os.path.join(CACHE_DIR, "ngl_" + name)
    if _download(NGL_URL.format(name=name), dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"NGL test map {name}"
    return None


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    PolNet templates are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    com = np.array(ndimage.center_of_mass(v))
    shift = np.array(v.shape) / 2 - 0.5 - com
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)
    dst0 = np.maximum((size - s) // 2, 0)
    n = np.minimum(s, size)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes


print("🧰 toolkit loaded:", ", ".join(["fetch_zoo_map", "fetch_ngl_map", "read_mrc", "prepare_map", "pearson",
                                      "show_orthoslices"]))

---
## 1 · Arrays: the grid of numbers behind every image

> 📍 **Where we are.** In [Module 00](00_welcome_cells_to_structures.ipynb) you met variables, lists, loops and functions,
> and you *looked at* density maps of the molecular zoo. Now we open the box: this module teaches the one data structure
> used by every cryo-EM/ET algorithm — the **NumPy array** — using real maps as examples.

### 1.1 Why not just use Python lists?

A Python **list** is a flexible container (it can hold numbers, text, anything). A NumPy **array** holds numbers of
one type, arranged in a regular grid, and does arithmetic on *all of them at once*. Watch what `* 2` means for each:

In [ ]:
# ── 1.1 Lists vs arrays ──
profile_list = [0.1, 0.3, 1.0, 0.4, 0.2, 0.9, 0.3]   # density along a line crossing a membrane (made-up values)
profile = np.array(profile_list)                     # the same numbers as a NumPy array

print("list  * 2 →", profile_list * 2)              # a list is REPEATED
print("array * 2 →", profile * 2)                   # an array is MULTIPLIED element by element
print("array + 1 →", profile + 1)
print("mean density:", profile.mean(), "| highest value:", profile.max(), "at position", profile.argmax())

🔍 **What just happened?** For a list, `* 2` means "glue two copies together"; for an array it means "double every
number". In science we almost always want the second meaning — that is why all our images and maps are arrays.
Notice also `argmax()`: the *position* (index) of the largest value. Positions count from **0**, so position 2 is the
third number.

### 1.2 Shape, dtype and memory

Every array has a **shape** (how many entries along each axis), a number of dimensions **`ndim`**, and a **`dtype`**
(the kind of number stored: `float32` = decimal number in 4 bytes, `int16` = whole number in 2 bytes, …).

| Object | Typical shape | Axis order in this course |
|---|---|---|
| a line profile | `(n,)` | `[x]` |
| a micrograph / projection / tilt image | `(ny, nx)` | `img[y, x]` (row, column) |
| a density map / subtomogram / tomogram | `(nz, ny, nx)` | `vol[z, y, x]` |
| a tilt series | `(n_tilts, ny, nx)` | `tilts[i, y, x]` |

⚠️ **Convention:** we always write `[z, y, x]` — the *last* index is x (columns). This matches how MRC files store data
(Section 5) and how NumPy lays out memory. Mixing up the order is the most common bug in this field (Section 6 shows
what it does to a map).

In [ ]:
# ── 1.2 Making arrays and asking them questions ──
blank_tomogram = np.zeros((64, 128, 128), dtype=np.float32)   # [z, y, x], all zeros
ramp = np.arange(0, 10, 2)                                   # 0, 2, 4, 6, 8 (stop value excluded!)
grid = np.linspace(-1, 1, 5)                                 # 5 evenly spaced values from -1 to 1 (inclusive)
image = np.arange(48).reshape(6, 8)                          # 0..47 rearranged into 6 rows × 8 columns

for name, a in [("blank_tomogram", blank_tomogram), ("ramp", ramp), ("grid", grid), ("image", image)]:
    print(f"{name:15s} shape={str(a.shape):14s} ndim={a.ndim}  dtype={str(a.dtype):8s} size={a.size:7d}  "
          f"memory={a.nbytes / 1e3:8.1f} kB")

# how much memory does a real tomogram need?
nz, ny, nx = 300, 1000, 1000
print(f"\nA {nx}×{ny}×{nz}-voxel tomogram in float32 needs {nz * ny * nx * 4 / 1e9:.1f} GB of memory.")

💡 Memory matters: a single binned cellular tomogram (10⁸–10⁹ voxels) can need gigabytes, which is why the field
works with **subtomograms** (small boxes, e.g. 32³ voxels, around each particle) and why Section 4's *binning* is used
everywhere.

### 1.3 Indexing and slicing

- `image[r, c]` → one number (row `r`, column `c`, both counted from 0). Negative indices count from the end: `-1` = last.
- `image[a:b]` → a **slice**: rows `a, a+1, …, b−1` (the stop value is excluded). Missing numbers mean "from the start" /
  "to the end"; `::2` means "every second one".
- `image[:, c]` → the whole column `c`; `image[r, :]` (or just `image[r]`) → the whole row `r`.

In [ ]:
# ── 1.3 Indexing and slicing a 6 × 8 "image" ──
print(image)
print("image[2, 3]   =", image[2, 3], "   (row 2, column 3)")
print("image[-1, -1] =", image[-1, -1], "  (last row, last column)")
print("image[1:4, 2:6] =\n", image[1:4, 2:6])
print("image[:, 0]   =", image[:, 0], "   (first column)")
print("image[::2, ::2] =\n", image[::2, ::2], "\n(every second row and column: a crude 'downsampling')")

fig, axes = plt.subplots(1, 3, figsize=(9, 2.6))
for ax, (sel, title) in zip(axes, [((2, 3), "image[2, 3]"), ((slice(1, 4), slice(2, 6)), "image[1:4, 2:6]"),
                                   ((slice(None), 0), "image[:, 0]")]):
    highlight = np.zeros_like(image, dtype=float)
    highlight[sel] = 1
    ax.imshow(image, cmap="Greys", alpha=0.35)
    ax.imshow(np.ma.masked_equal(highlight, 0), cmap="autumn", alpha=0.8)
    for (r, c), val in np.ndenumerate(image):
        ax.text(c, r, val, ha="center", va="center", fontsize=7)
    ax.set_title(title, fontsize=9, family="monospace")
    ax.set_xlabel("column (x)"); ax.set_ylabel("row (y)")
plt.tight_layout(); plt.show()

⚠️ **Pitfall — a slice is a *view*, not a copy.** A slice shares memory with the original array. If you change the
slice, you change the original. When you want an independent piece (e.g. to "extract a subtomogram" and then modify
it), call `.copy()`.

In [ ]:
# ── 1.3b Views versus copies ──
original = np.arange(10)
view = original[2:5]          # a window onto the same memory
safe = original[2:5].copy()   # an independent copy
view[:] = -1                  # write into the view...
safe[:] = 99                  # ...and into the copy
print("original after editing the view and the copy:", original)
print("→ the view changed the original; the copy did not.")

### 1.4 Vectorised maths, reductions and speed

**Vectorised** means: write the operation once for the whole array instead of looping over elements. A **reduction**
summarises many numbers into fewer: `sum`, `mean`, `std` (standard deviation), `min`, `max`. With `axis=` you reduce
along one direction only — for a 3D map, `vol.sum(axis=0)` adds up along z and leaves a 2D `[y, x]` image. Remember
that one: it is how a microscope makes a projection image (Section 3).

In [ ]:
# ── 1.4 A Python loop versus one NumPy call ──
values = rng.normal(size=1_000_000)          # one million random numbers

t0 = time.perf_counter()
total = 0.0
for v in values:                              # pure-Python loop
    total += v
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
total_np = values.sum()                       # vectorised
t_np = time.perf_counter() - t0
print(f"loop sum  = {total:.4f} in {t_loop * 1e3:7.1f} ms")
print(f"numpy sum = {total_np:.4f} in {t_np * 1e3:7.1f} ms  → about {t_loop / max(t_np, 1e-9):.0f}× faster")

stack = rng.normal(size=(4, 3, 5))            # e.g. a tiny volume [z=4, y=3, x=5]
print("\nshape of stack:", stack.shape)
print("stack.sum(axis=0).shape =", stack.sum(axis=0).shape, " (z removed → a [y, x] image)")
print("stack.mean(axis=(1, 2)) =", np.round(stack.mean(axis=(1, 2)), 3), " (one mean per z-slice)")

### 1.5 Boolean masks: asking yes/no questions of every pixel

A comparison such as `img > 0.5` returns an array of `True`/`False` of the same shape — a **mask**. Masks let you count
("how many voxels are inside the molecule?"), select (`img[mask]` gives the values where the mask is True) and edit
(`img[mask] = 0`). Since `True` counts as 1, `mask.sum()` counts the True entries.

In [ ]:
# ── 1.5 Masks ──
noisy = rng.normal(0, 1, size=(6, 8))
mask = noisy > 1.0                                   # which pixels are brighter than 1?
print(mask.astype(int))
print("pixels above 1.0:", mask.sum(), "of", mask.size, f"({100 * mask.mean():.0f} %)")
print("their mean value:", noisy[mask].mean().round(3))
rows, cols = np.nonzero(mask)                        # coordinates of the True pixels
print("their (row, column) positions:", list(zip(rows.tolist(), cols.tolist())))

### 1.6 Broadcasting: drawing a particle with one line of maths

**Broadcasting** lets NumPy combine arrays of different shapes by stretching size-1 axes. A column of y-values (shape
`(ny, 1)`) combined with a row of x-values (shape `(1, nx)`) gives a full `(ny, nx)` grid — perfect for evaluating a
formula at every pixel. For example, a round "particle" of radius $\sigma$ centred at $(x_0, y_0)$:

$$I(x, y) = \exp\!\left(-\frac{(x - x_0)^2 + (y - y_0)^2}{2\sigma^2}\right)$$

*In words:* brightness is 1 at the centre and fades smoothly with distance — a **Gaussian blob**, the simplest model of
a blurred molecule. In 3D, `np.meshgrid(..., indexing="ij")` builds the coordinate grids in `[z, y, x]` order.

In [ ]:
# ── 1.6 Broadcasting: a 2D Gaussian particle and a 3D hollow sphere (a toy vesicle) ──
ny, nx = 40, 60
y = np.arange(ny)[:, None]           # shape (40, 1) — a column
x = np.arange(nx)[None, :]           # shape (1, 60) — a row
print("shapes:", y.shape, "+", x.shape, "→", (x + y).shape)
particle = np.exp(-((x - 20)**2 + (y - 15)**2) / (2 * 4.0**2))   # Gaussian blob at (x=20, y=15), σ = 4 px

n = 48
zz, yy, xx = np.meshgrid(np.arange(n), np.arange(n), np.arange(n), indexing="ij")   # [z, y, x] grids
r = np.sqrt((zz - n / 2)**2 + (yy - n / 2)**2 + (xx - n / 2)**2)                   # distance from the centre
vesicle = np.exp(-((r - 16)**2) / (2 * 1.5**2))       # a thin spherical shell: a lipid vesicle of radius 16 voxels

fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))
axes[0].imshow(particle); axes[0].set_title("2D Gaussian particle")
axes[0].set_xlabel("x (px)"); axes[0].set_ylabel("y (px)")
axes[1].imshow(vesicle[n // 2]); axes[1].set_title("vesicle: slice through the middle")
axes[2].imshow(vesicle.sum(axis=0)); axes[2].set_title("vesicle: projection (sum over z)")
for ax in axes[1:]:
    ax.set_xlabel("x (voxels)"); ax.set_ylabel("y (voxels)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** One line of arithmetic, evaluated at every grid point at once, drew a particle. The vesicle
looks like a **ring in a slice** but like a **disc with a bright rim in a projection** — because the beam passes through
a lot of membrane at the edge of the sphere. That is exactly how vesicles look in cryo-EM images.

✏️ **Your turn.** In the cell above, make the particle elongated: replace the single σ by different widths in x and y,
e.g. `np.exp(-((x - 20)**2 / (2 * 8.0**2) + (y - 15)**2 / (2 * 3.0**2)))`. Then make the vesicle membrane thicker
(change `1.5` to `3`). *Hint: thickness ≈ 2.35 σ voxels (the "full width at half maximum").*

---
## 2 · 2D images: micrographs, colormaps and contrast

An electron micrograph is a 2D array of electron counts. We display arrays with `plt.imshow`, which maps numbers to
colours through a **colormap**. Two rules for this course:

- Use **`gray`** for densities and micrographs (brightness = amount of matter; no false structure from colours).
- Use **`viridis`** for derived maps (correlations, resolution, errors), where colour should mean a value. Always add a
  **colorbar** when the numbers matter.

Let us make a perfect, noise-free "micrograph" of one 20S proteasome by projecting its map (the beam adds up the
density along its path — Section 3), and then a realistic noisy one.

In [ ]:
# ── 2.1 A projection image and a noisy version of it ──
prot_raw, prot_voxel, prot_src = fetch_zoo_map("3j9i")
print(f"20S proteasome map: shape {prot_raw.shape}, {prot_voxel:.0f} Å/voxel  ← {prot_src}")
side_view = prepare_map(prot_raw, size=40, sigma=0.5).sum(axis=0)[6:34, 6:34]   # sum along z, central 28 × 28 nm
micrograph = side_view + rng.normal(0, side_view.std() * 1.0, side_view.shape)   # noise as strong as the signal

fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.8))
axes[0].imshow(side_view, cmap="gray"); axes[0].set_title("clean, cmap='gray'")
im = axes[1].imshow(side_view, cmap="viridis"); axes[1].set_title("clean, cmap='viridis'")
plt.colorbar(im, ax=axes[1], fraction=0.046, label="projected density (a.u.)")
axes[2].imshow(micrograph, cmap="gray"); axes[2].set_title("noisy (SNR = 1)")
lo, hi = np.percentile(micrograph, [2, 98])
axes[3].imshow(micrograph, cmap="gray", vmin=lo, vmax=hi); axes[3].set_title("noisy, contrast 2–98 %")
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In this map the proteasome happens to lie on its side (tilted in the image plane), so the
projection shows the barrel from the side, with faint stripes across it — its four stacked rings of seven subunits. With
noise as strong as the signal (signal-to-noise ratio, **SNR** = variance of signal / variance of noise = 1) the shape is
still visible; real cryo-EM images are 10–100× noisier. Setting `vmin`/`vmax` to the 2nd and 98th **percentiles** (the
values below which 2 % and 98 % of pixels fall) ignores a few extreme pixels and stretches the useful grey range —
the same "auto-contrast" that IMOD, ChimeraX and Fiji perform.

### 2.2 Histograms: the distribution of pixel values

A **histogram** counts how many pixels fall in each range of values. It is the first thing to look at when an image
"looks wrong" (all black? a few huge outliers?).

In [ ]:
# ── 2.2 Histograms of the clean and noisy images ──
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.hist(side_view.ravel(), bins=60, alpha=0.6, label="clean projection")
ax.hist(micrograph.ravel(), bins=60, alpha=0.6, label="noisy micrograph")
for v, lab in [(lo, "2 %"), (hi, "98 %")]:
    ax.axvline(v, color="k", ls="--", lw=1)
    ax.text(v, ax.get_ylim()[1] * 0.9, f" {lab}", fontsize=8)
ax.set_xlabel("pixel value (a.u.)"); ax.set_ylabel("number of pixels"); ax.set_yscale("log")
ax.set_title("Histogram: most pixels are background; noise spreads values out")
ax.legend(fontsize=8); plt.show()

---
## 3 · 3D volumes: slices, projections, thresholds and surfaces

Now the main character: the bacterial **70S ribosome** (PDB 4V4R), the protein factory of the cell and the most-studied
particle in cellular cryo-ET. Its map from PolNet is a 3D array with 10 Å (= 1 nm) voxels.

In [ ]:
# ── 3.1 Load the ribosome map and look at the raw numbers ──
ribo_raw, ribo_voxel, ribo_src = fetch_zoo_map("4v4r")
print(f"source: {ribo_src}")
print(f"shape [z, y, x] = {ribo_raw.shape}, dtype = {ribo_raw.dtype}, voxel = {ribo_voxel:.0f} Å "
      f"→ box = {ribo_raw.shape[2] * ribo_voxel / 10:.0f} nm wide")
print(f"min = {ribo_raw.min():.1f}, max = {ribo_raw.max():.1f}, mean = {ribo_raw.mean():.2f}; "
      f"voxels with any density: {(ribo_raw > 0).sum()} of {ribo_raw.size}")
c = np.array(ribo_raw.shape) // 2
print("\nA 5 × 5 patch of numbers from the central z-slice (this is what a map *is*):")
print(np.round(ribo_raw[c[0], c[1] - 2:c[1] + 3, c[2] - 2:c[2] + 3], 1))

### 3.2 Orthogonal slices and flipping through a volume

A **slice** is a 2D cut: `vol[k]` is the xy-plane at height z = k; `vol[:, j, :]` is the xz-plane at y = j;
`vol[:, :, i]` is the yz-plane at x = i. The three central slices are called **orthoslices** — the standard first look at
any map or tomogram (the toolkit's `show_orthoslices`). Paging through all z-slices is how people inspect tomograms in
IMOD or napari.

In [ ]:
# ── 3.2 Orthoslices and a gallery of z-slices ──
ribo = prepare_map(ribo_raw, size=48, sigma=0.5)       # centred, 48³, scaled to [0, 1]
show_orthoslices(ribo, title="70S ribosome")
plt.show()

z_values = np.arange(8, 41, 4)                          # every 4th slice through the molecule
fig, axes = plt.subplots(1, len(z_values), figsize=(11, 1.7))
for ax, k in zip(axes, z_values):
    ax.imshow(ribo[k], vmin=0, vmax=1)
    ax.set_title(f"z = {k}", fontsize=8); ax.axis("off")
fig.suptitle("Paging through the ribosome, one 1-nm slice at a time", fontsize=9, y=1.05)
plt.show()

### 3.3 Projections: what the microscope actually records

In the **projection approximation** (Module 06), an electron image is the *sum of the density along the beam*:

$$P(y, x) = \sum_{z} V(z, y, x)$$

*In words:* each image pixel adds up all the voxels stacked above it. In NumPy that is just `vol.sum(axis=0)`. Summing
along y or x gives views from other directions. A **maximum-intensity projection** (`vol.max(axis=0)`), common in light
microscopy, keeps only the brightest voxel along each ray instead — useful for display, but *not* what an electron
microscope measures.

In [ ]:
# ── 3.3 Slice versus projection, from three directions ──
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for j, (axis, lab) in enumerate([(0, "along z (beam)"), (1, "along y"), (2, "along x")]):
    middle = np.take(ribo, ribo.shape[axis] // 2, axis=axis)   # the central slice perpendicular to this axis
    axes[0, j].imshow(middle); axes[0, j].set_title(f"central slice, normal to {lab.split()[1]}", fontsize=8)
    axes[1, j].imshow(ribo.sum(axis=axis)); axes[1, j].set_title(f"projection {lab}", fontsize=8)
    axes[2, j].imshow(ribo.max(axis=axis)); axes[2, j].set_title(f"max-intensity proj. {lab}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

print("Projections conserve the total: sum of volume =", round(float(ribo.sum()), 2),
      "| sum of each projection =", [round(float(ribo.sum(axis=a).sum()), 2) for a in range(3)])

🔍 **What just happened?** A slice shows a thin 1-nm section (where the RNA and protein are at that height); a
projection shows *everything* along the beam superimposed, so internal details overlap. Projections conserve the total
density — no matter is lost, it is only stacked. This superposition is why single 2D images are ambiguous and why we
need tomography (Module 07) or many views (Module 06) to recover 3D.

### 3.4 Thresholds, contour levels and isosurfaces

3D viewers such as ChimeraX show a map as a surface: every voxel above a chosen **threshold** (contour level) is "inside"
the molecule. The choice matters — a low threshold makes the molecule look fat (noise and blur included), a high one
makes it fall apart into islands. Let us measure the enclosed volume and the number of separate pieces
(`ndimage.label` finds **connected components**, groups of touching voxels) as the threshold changes, and compare with
the volume expected from the mass (Module 00): $V = M / (0.81\ \text{Da/Å}^3)$.
Then we draw the surface itself at two contour levels.

In [ ]:
# ── 3.4 Volume and number of pieces as a function of the threshold ──
expected_nm3 = ZOO["4v4r"]["mass_kDa"] * 1000 / 0.81 / 1000        # Da → Å³ → nm³ (= voxels at 1 nm)
thresholds = np.linspace(0.02, 0.9, 45)
volumes = np.array([(ribo > t).sum() for t in thresholds])          # each voxel is 1 nm³
pieces = np.array([ndimage.label(ribo > t)[1] for t in thresholds])
t_mass = thresholds[np.argmin(np.abs(volumes - expected_nm3))]
print(f"expected volume from 2,277 kDa ≈ {expected_nm3:.0f} nm³ → threshold ≈ {t_mass:.2f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(thresholds, volumes, color="tab:blue", label="enclosed volume")
ax.axhline(expected_nm3, color="tab:blue", ls=":", lw=1)
ax.axvline(t_mass, color="k", ls="--", lw=1, label=f"mass-based threshold ({t_mass:.2f})")
ax.set_xlabel("threshold (fraction of maximum density)"); ax.set_ylabel("volume inside (nm³)", color="tab:blue")
ax2 = ax.twinx()
ax2.step(thresholds, pieces, color="tab:red", where="mid", label="number of pieces")
ax2.set_ylabel("connected pieces", color="tab:red")
ax.set_title("Choosing a contour level for the 70S ribosome")
ax.legend(loc="upper right", fontsize=8); plt.show()

In [ ]:
# ── 3.5 An isosurface rendering of the ribosome at two contour levels ──
# "Marching cubes" (scikit-image) turns a 3D array + a threshold into a mesh of triangles — the same idea ChimeraX
# uses to draw map surfaces (interactively, and much prettier).
from skimage import measure

fig = plt.figure(figsize=(8, 3.8))
for k, level in enumerate([0.15, t_mass]):
    verts, faces, _, _ = measure.marching_cubes(ribo, level=level)       # vertices in [z, y, x] voxel units
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.plot_trisurf(verts[:, 2], verts[:, 1], faces, verts[:, 0], color="goldenrod", lw=0, shade=True)
    ax.set_title(f"threshold {level:.2f}: {(ribo > level).sum()} nm³, {len(faces)} triangles", fontsize=8)
    ax.set_xlabel("x (nm)", fontsize=7); ax.set_ylabel("y (nm)", fontsize=7); ax.set_zlabel("z (nm)", fontsize=7)
    ax.tick_params(labelsize=6); ax.view_init(elev=20, azim=-60)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The same array gives a fat or a lean molecule depending on one number. Published figures
should state the contour level (often chosen so the enclosed volume matches the mass, or as a multiple of the noise
standard deviation). The ribosome contains a lot of RNA, which is denser than protein, so the 0.81 Da/Å³ rule is only
approximate here.

✏️ **Your turn.** Repeat 3.4–3.5 for the 20S proteasome: replace `ribo` by
`prepare_map(prot_raw, size=48, sigma=0.5)` and `"4v4r"` by `"3j9i"`. At which threshold does the barrel's central
channel become visible from the top?

---
## 4 · Units: voxel size, ångströms and binning

An array index is not a physical distance. The **voxel size** (also *pixel size* or *sampling*, in Å/voxel) converts
one to the other:

$$\text{distance (Å)} = \text{distance (voxels)} \times \text{voxel size (Å/voxel)}$$

Our zoo maps use 10 Å voxels (1 nm), typical of binned cellular tomograms. Single-particle maps are much finer. Let us
load a **real single-particle cryo-EM map** of **β-galactosidase** (the lactose-cutting enzyme, and a favourite test
specimen of cryo-EM labs), shipped as test data with the NGL molecular viewer and sampled at 3.54 Å/voxel.

**Binning** by a factor *b* averages each b×b×b block of voxels into one: the array gets *b*³ times smaller and the voxel
size *b* times larger. It speeds up everything — but no detail smaller than about **2 voxels** can survive (the
**Nyquist limit**: to see a repeating pattern you need at least two samples per period; Module 03 explains why).

In [ ]:
# ── 4.1 Load a real cryo-EM map (β-galactosidase) ──
bgal = fetch_ngl_map("betaGal.mrc")
if bgal is None:                                       # offline fallback: a stand-in shape
    bgal = (np.kron(procedural_molecule("1bxn", size=50), np.ones((2, 2, 2))), 3.54, "procedural stand-in (offline)")
bgal_map, bgal_voxel, bgal_src = bgal
print(f"source: {bgal_src}; shape {bgal_map.shape}; voxel {bgal_voxel:.2f} Å "
      f"→ box {bgal_map.shape[2] * bgal_voxel / 10:.1f} nm; values {bgal_map.min():.3f} … {bgal_map.max():.3f}")


def bin_volume(vol: np.ndarray, b: int) -> np.ndarray:
    """Average non-overlapping b×b×b blocks (crop so each side is a multiple of b)."""
    nz, ny, nx = (np.array(vol.shape) // b) * b
    v = vol[:nz, :ny, :nx]
    # The reshape trick: e.g. (96, 96, 96) → (48, 2, 48, 2, 48, 2) puts each pair of neighbours along an axis into
    # its own small axis; averaging over axes 1, 3 and 5 then averages every 2×2×2 block. No loops needed.
    return v.reshape(nz // b, b, ny // b, b, nx // b, b).mean(axis=(1, 3, 5))

In [ ]:
# ── 4.2 Bin β-galactosidase by 1, 2, 4 and 8 ──
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j, b in enumerate([1, 2, 4, 8]):
    vb = bin_volume(bgal_map, b) if b > 1 else bgal_map
    vs = bgal_voxel * b
    k = vb.shape[0] // 2
    axes[0, j].imshow(vb[k]); axes[0, j].set_title(f"bin {b}: {vb.shape[0]}³ @ {vs:.2f} Å\ncentral slice", fontsize=8)
    axes[1, j].imshow(vb.sum(axis=0)); axes[1, j].set_title(f"projection; finest detail ≈ {2 * vs:.0f} Å", fontsize=8)
    for ax in axes[:, j]:
        ax.axis("off")
    print(f"bin {b}: shape {vb.shape}, voxel {vs:5.2f} Å, memory {vb.astype(np.float32).nbytes / 1e6:6.3f} MB, "
          f"Nyquist limit {2 * vs:5.1f} Å")
fig.suptitle("Real map of β-galactosidase: binning trades detail for speed", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At 3.54 Å/voxel the slice shows fine internal texture (secondary-structure-sized features);
at 28 Å/voxel only the overall outline survives. Cellular tomograms are usually analysed at 10–20 Å/voxel —
enough to recognise a ribosome's shape, not its helices. *Choosing the voxel size is choosing which biology you can see.*

✏️ **Your turn.** Measure β-galactosidase: threshold the bin-2 map at 30 % of
its maximum, find the extent of the True voxels along x (`np.nonzero(mask)[2]` gives their x-indices; take
max − min + 1), and convert it to nm with the voxel size. Then do the same along y and z (indices `[1]` and `[0]`), and
try thresholds of 10 % and 50 %: how much does the "size" of a molecule depend on the contour level?

In [ ]:
# ── ✏️ 4.3 Your turn: measure a molecule in nanometres ──
vb2 = bin_volume(bgal_map, 2)
mask_b = vb2 > 0.3 * vb2.max()
xs = np.nonzero(mask_b)[2]
extent_vox = xs.max() - xs.min() + 1
print(f"extent along x: {extent_vox} voxels × {2 * bgal_voxel:.2f} Å = {extent_vox * 2 * bgal_voxel / 10:.1f} nm")

---
## 5 · Opening a density-map file byte by byte (MRC format)

Maps, tomograms, tilt series and micrographs are stored in **MRC files** (extensions `.mrc`, `.map`, `.rec`, `.st`;
the community standard *MRC2014*). A file is just a long sequence of **bytes** (numbers 0–255). An MRC file is:

1. a **header** of exactly **1024 bytes** = 256 "words" of 4 bytes, describing the data;
2. an optional **extended header** (length given in word 24, often 0);
3. the **data**: `nx × ny × nz` numbers, with **x changing fastest**, then y, then z.

The words we need (word *n* starts at byte 4·(*n*−1)):

| word(s) | bytes | name | meaning |
|---|---|---|---|
| 1–3 | 0–11 | NX, NY, NZ | number of columns (x), rows (y), sections (z) |
| 4 | 12–15 | MODE | data type: 0 = int8, 1 = int16, **2 = float32**, 6 = uint16 |
| 8–10 | 28–39 | MX, MY, MZ | number of grid intervals in the unit cell (usually = NX, NY, NZ) |
| 11–13 | 40–51 | cell dimensions | size of the box in Å (so voxel size = cell / MX) |
| 17–19 | 64–75 | MAPC, MAPR, MAPS | which axis is columns/rows/sections (normally 1, 2, 3 = x, y, z) |
| 20–22 | 76–87 | DMIN, DMAX, DMEAN | minimum, maximum, mean value |
| 24 | 92–95 | NSYMBT | length of the extended header in bytes |
| 53 | 208–211 | MAP | the text `"MAP "` — identifies the file type |
| 54 | 212–215 | MACHST | "machine stamp": byte order (0x44 0x44 or 0x44 0x41 = little-endian) |

Python's built-in `struct` module turns bytes into numbers: `struct.unpack("<3i", raw[0:12])` reads three
little-endian (`<`) 4-byte integers (`i`); `f` means a 4-byte float.

In [ ]:
# ── 5.1 Read the first bytes of the ribosome map file ──
mrc_path = os.path.join(CACHE_DIR, "polnet_10A_4v4r.mrc")
if os.path.exists(mrc_path):
    raw = open(mrc_path, "rb").read()
else:   # offline: build a valid MRC file in memory from the stand-in map (Section 5.4 explains each line)
    nz_, ny_, nx_ = ribo_raw.shape
    hdr = bytearray(1024)
    struct.pack_into("<4i", hdr, 0, nx_, ny_, nz_, 2)
    struct.pack_into("<3i", hdr, 28, nx_, ny_, nz_)
    struct.pack_into("<3f", hdr, 40, nx_ * 10.0, ny_ * 10.0, nz_ * 10.0)
    struct.pack_into("<3i", hdr, 64, 1, 2, 3)
    hdr[208:212] = b"MAP "
    raw = bytes(hdr) + ribo_raw.astype(np.float32).tobytes()
print(f"file size: {len(raw):,} bytes")
print("first 16 bytes as numbers 0–255:", list(raw[:16]))
print("the same bytes in hexadecimal  :", raw[:16].hex(" "))

In [ ]:
# ── 5.2 Decode the header words ──
nx_f, ny_f, nz_f, mode = struct.unpack("<4i", raw[0:16])
mx, my, mz = struct.unpack("<3i", raw[28:40])
cell = struct.unpack("<3f", raw[40:52])
mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
dmin, dmax, dmean = struct.unpack("<3f", raw[76:88])
nsymbt = struct.unpack("<i", raw[92:96])[0]
header_table = [("NX, NY, NZ", (nx_f, ny_f, nz_f)), ("MODE", mode), ("MX, MY, MZ", (mx, my, mz)),
                ("cell (Å)", tuple(round(c_, 1) for c_ in cell)), ("MAPC/R/S", (mapc, mapr, maps)),
                ("DMIN/DMAX/DMEAN", (round(dmin, 2), round(dmax, 2), round(dmean, 3))),
                ("NSYMBT", nsymbt), ("MAP", raw[208:212]), ("MACHST", raw[212:216].hex(" "))]
for name, value in header_table:
    print(f"  {name:16s} {value}")
voxel_size = cell[0] / mx
print(f"→ voxel size = cell / MX = {cell[0]:.1f} / {mx} = {voxel_size:.2f} Å")

💡 The header says MODE 2 (float32, 4 bytes per value), so the data block must be `NX·NY·NZ·4` bytes long, starting
right after the 1024-byte header (plus `NSYMBT` bytes of extended header). Because x changes fastest in the file, the
natural NumPy shape is `(NZ, NY, NX)` — the reason behind our `[z, y, x]` convention.

In [ ]:
# ── 5.3 Decode the data block and compare with the toolkit's read_mrc ──
dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
n_values = nx_f * ny_f * nz_f
print(f"expected file size = 1024 + {nsymbt} + {n_values} × {np.dtype(dtype).itemsize} = "
      f"{1024 + nsymbt + n_values * np.dtype(dtype).itemsize:,} bytes (actual {len(raw):,})")
data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=n_values).reshape(nz_f, ny_f, nx_f)
vol_toolkit, vs_toolkit = read_mrc(raw)
print("our array:", data.shape, "| toolkit:", vol_toolkit.shape, f"@ {vs_toolkit:.1f} Å")
print("identical to the toolkit's result:", np.array_equal(data, vol_toolkit))
print(f"header DMAX = {dmax:.2f} vs data max = {data.max():.2f}")
assert np.array_equal(data, vol_toolkit), "our byte-level reader should agree with read_mrc"

### 5.4 Writing your own MRC file

To look at a map you computed in ChimeraX, IMOD or napari, you need to *write* an MRC file. The function below fills a
1024-byte header with the essential words and appends the data. (The `mrcfile` Python package does this more
completely; we write it by hand once so nothing is magic.)

In [ ]:
# ── 5.4 A minimal MRC2014 writer, and a round-trip test ──
def write_mrc(path: str, vol: np.ndarray, voxel_A: float, label: str = "BiologyToResearchV3") -> None:
    """Write a volume[z, y, x] as a float32 (mode 2) MRC2014 file that ChimeraX/IMOD can open."""
    vol = np.ascontiguousarray(vol, dtype=np.float32)
    nz, ny, nx = vol.shape
    hdr = bytearray(1024)                                                   # 256 words, all zero to start
    struct.pack_into("<4i", hdr, 0, nx, ny, nz, 2)                          # words 1-4: size and mode 2
    struct.pack_into("<3i", hdr, 28, nx, ny, nz)                            # words 8-10: sampling MX, MY, MZ
    struct.pack_into("<3f", hdr, 40, nx * voxel_A, ny * voxel_A, nz * voxel_A)  # words 11-13: cell size (Å)
    struct.pack_into("<3f", hdr, 52, 90.0, 90.0, 90.0)                      # words 14-16: cell angles
    struct.pack_into("<3i", hdr, 64, 1, 2, 3)                               # words 17-19: x, y, z axis order
    struct.pack_into("<3f", hdr, 76, vol.min(), vol.max(), vol.mean())      # words 20-22
    struct.pack_into("<i", hdr, 88, 1)                                      # word 23: space group 1 = a volume
    struct.pack_into("<i", hdr, 108, 20140)                                 # word 28: format version 2014
    hdr[208:212] = b"MAP "                                                  # word 53
    hdr[212:216] = bytes([0x44, 0x44, 0, 0])                                # word 54: little-endian
    struct.pack_into("<f", hdr, 216, vol.std())                             # word 55: RMS deviation
    struct.pack_into("<i", hdr, 220, 1)                                     # word 56: one text label
    hdr[224:224 + 80] = label.encode("ascii")[:80].ljust(80)                # words 57-76: the label
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "wb") as f:
        f.write(bytes(hdr))
        f.write(vol.tobytes())


out_dir = "outputs_01"
write_mrc(os.path.join(out_dir, "ribosome_48cube_10A.mrc"), ribo, voxel_A=10.0, label="70S ribosome, prepared")
back, back_vs = read_mrc(os.path.join(out_dir, "ribosome_48cube_10A.mrc"))
print("written and read back:", back.shape, f"@ {back_vs} Å; identical:", np.array_equal(back, ribo))
assert np.array_equal(back, ribo) and back_vs == 10.0

🔍 **What just happened?** You decoded a scientific file format from raw bytes, checked it against a library reader,
and wrote a valid file of your own. Download `outputs_01/ribosome_48cube_10A.mrc` (Colab: the folder icon on the left)
and open it in [ChimeraX](https://www.cgl.ucsf.edu/chimerax/) if you have it installed.

⚠️ **Pitfalls with real files:** some programs write MAPC/MAPR/MAPS ≠ 1, 2, 3 (axes permuted — `read_mrc` handles it);
some files are big-endian (machine stamp `0x11 0x11`); tomograms and raw camera data often use integer modes 0, 1 or 6; and
the **origin** (words 50–52) places the box in space — ignore it and two maps that "should" overlap won't.

---
## 6 · From atoms to density

Where do density maps of known structures come from? From **atomic models** in the Protein Data Bank (PDB). A PDB file
lists every atom with its 3D coordinates in ångströms. If we "paint" each atom into a grid of voxels and blur it to the
resolution we want, we obtain a **simulated map** — exactly how the PolNet templates, and the reference maps used by
template matching and atomic-model fitting, are made.

We use **RuBisCO** (PDB 1BXN, from the bacterium *Cupriavidus necator*), the carbon-fixing enzyme of photosynthesis. We
download the same PDB file that PolNet used (pinned version), with a synthetic fallback if you are offline.

### 6.1 A PDB file is fixed-column text

Each atom is one line starting with `ATOM` (standard residues) or `HETATM` (other molecules: ions, ligands, water).
Fields sit at **fixed columns** (counting columns from 1, as the PDB format documentation does):

| columns | field | example |
|---|---|---|
| 1–6 | record type | `ATOM  ` |
| 13–16 | atom name | ` CA ` (alpha carbon) |
| 18–20 | residue name | `TYR` |
| 22 | chain identifier | `A` |
| 23–26 | residue number | `  22` |
| 31–38, 39–46, 47–54 | x, y, z (Å) | `-62.356` |
| 77–78 | element symbol | ` C` |

⚠️ Python counts from 0 and excludes the stop, so columns 31–38 are `line[30:38]`.

In [ ]:
# ── 6.1 Download the PDB file (pinned PolNet copy) with an offline fallback ──
PDB_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
           "73e613cc9162d26c7fd740ce3299bcb280a3e3b6/data/templates/pdbs/1bxn.pdb")


def synthetic_pdb_lines(n_res: int = 2000, seed: int = 0) -> list[str]:
    """Offline fallback: fake ATOM records (a blobby two-domain 'protein'; NOT a real structure)."""
    r = np.random.default_rng(seed)
    lines, serial = [], 1
    centres = np.concatenate([r.normal(0, 22, (n_res // 2, 3)), r.normal(0, 14, (n_res // 2, 3)) + [40, 0, 0]])
    for i, c in enumerate(centres):
        for name, el in [("N", "N"), ("CA", "C"), ("C", "C"), ("O", "O"), ("CB", "C")]:
            x_, y_, z_ = c + r.normal(0, 1.2, 3)
            lines.append(f"ATOM  {serial:5d}  {name:<3s} ALA A{i % 9999 + 1:4d}    "
                         f"{x_:8.3f}{y_:8.3f}{z_:8.3f}  1.00  0.00          {el:>2s}")
            serial += 1
    return lines


pdb_path = os.path.join(CACHE_DIR, "polnet_pdb_1bxn.pdb")
if _download(PDB_URL, pdb_path):
    pdb_lines = open(pdb_path).read().splitlines()
    pdb_source = "PDB 1BXN (PolNet pinned copy)"
else:
    pdb_lines = synthetic_pdb_lines()
    pdb_source = "synthetic atoms (offline fallback — not a real protein)"
print(f"{pdb_source}: {len(pdb_lines):,} lines")
print("first header lines:")
for line in pdb_lines[:3]:
    print("  ", line)
atom_lines = [line for line in pdb_lines if line.startswith(("ATOM  ", "HETATM"))]
print("\ntwo atom records, with a column ruler:")
print("   " + "".join(str(i % 10) for i in range(1, 81)))
for line in atom_lines[:2]:
    print("   " + line)

In [ ]:
# ── 6.2 Parse atoms by fixed columns: keep ATOM/HETATM lines and cut each field out of its columns ──
def parse_pdb_atoms(lines: list[str]) -> dict:
    """Extract coordinates (Å), element, chain, residue number and record type from ATOM/HETATM lines."""
    xyz, element, chain, resnum, record = [], [], [], [], []
    for line in lines:
        if not line.startswith(("ATOM  ", "HETATM")):
            continue
        xyz.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])   # columns 31-38, 39-46, 47-54
        el = line[76:78].strip() if len(line) >= 78 else ""                       # columns 77-78
        element.append(el or line[12:14].strip()[0])                                # fallback: atom name
        chain.append(line[21])                                                      # column 22
        resnum.append(int(line[22:26]))                                             # columns 23-26
        record.append(line[:6].strip())
    return dict(xyz=np.array(xyz), element=np.array(element), chain=np.array(chain),
                resnum=np.array(resnum), record=np.array(record))


atoms = parse_pdb_atoms(pdb_lines)
xyz = atoms["xyz"]
print(f"{len(xyz):,} atoms; coordinate array shape {xyz.shape} (one row per atom: x, y, z in Å)")
elements, counts = np.unique(atoms["element"], return_counts=True)
print("atoms per element:", dict(zip(elements.tolist(), counts.tolist())))
chains, ch_counts = np.unique(atoms["chain"][atoms["record"] == "ATOM"], return_counts=True)
print("protein atoms per chain:", dict(zip(chains.tolist(), ch_counts.tolist())))
print("extent (max − min) along x, y, z:", np.round(np.ptp(xyz, axis=0), 1), "Å")

🧫 **Reading the biology out of the numbers.** RuBisCO in plants and bacteria such as *C. necator* is a "form I"
enzyme built from **large (L)** and **small (S)** subunits. In 1BXN the chains A, C, E, G each have ~3,470 atoms (large
subunits, ≈ 450 residues) and chains I, J, K, L each ~1,060 atoms (small subunits, ≈ 130 residues). That is **L₄S₄** —
only *half* of the biological L₈S₈ enzyme. The file holds the crystal's **asymmetric unit** (the smallest piece from
which the crystal is built); the full enzyme is generated by applying a symmetry operation listed in the file's
`REMARK 350` lines. *Whenever you use a PDB entry, check whether the file is the biological assembly.* Our zoo's
"RuBisCO" (and its listed 280 kDa) is this L₄S₄ half.

In [ ]:
# ── 6.3 Look at the atoms: one dot per atom, coloured by chain ──
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ch in np.unique(atoms["chain"]):
    sel = atoms["chain"] == ch
    for ax, (i, j, lab) in zip(axes, [(0, 1, ("x (Å)", "y (Å)")), (0, 2, ("x (Å)", "z (Å)"))]):
        ax.scatter(xyz[sel, i], xyz[sel, j], s=0.3, alpha=0.4, label=f"chain {ch}")
        ax.set_xlabel(lab[0]); ax.set_ylabel(lab[1]); ax.set_aspect("equal")
axes[0].set_title("atoms seen down z"); axes[1].set_title("atoms seen down y")
axes[1].legend(markerscale=15, fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout(); plt.show()

### 6.4 Mass from atoms

Two quick estimates of molecular mass (1 dalton, Da ≈ the mass of a hydrogen atom):

1. **Residue count × 110 Da** — the average mass of an amino-acid residue in a protein.
2. **Sum of atomic masses** — C 12.011, N 14.007, O 15.999, S 32.06, P 30.974 Da. X-ray models usually omit hydrogen
   atoms (too light to see), so this underestimates the true mass by roughly 7–8 %.

In [ ]:
# ── 6.4 Estimate the mass of the L4S4 complex ──
ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06}
is_protein = atoms["record"] == "ATOM"
residues = {(c_, r_) for c_, r_ in zip(atoms["chain"][is_protein], atoms["resnum"][is_protein])}
mass_residues_kDa = len(residues) * 110 / 1000
mass_atoms_kDa = sum(ATOMIC_MASS.get(e, 12.0) for e in atoms["element"][is_protein]) / 1000
print(f"modelled residues: {len(residues):,}  →  × 110 Da ≈ {mass_residues_kDa:.0f} kDa")
print(f"sum of heavy-atom masses ≈ {mass_atoms_kDa:.0f} kDa (no hydrogens)")
print(f"value listed in our ZOO table (from PolNet): {ZOO['1bxn']['mass_kDa']} kDa")

🔍 **What just happened?** Both estimates land a little below the listed mass. The gap is real biology and real
crystallography: flexible chain ends that were not visible in the crystal are missing from the model, and so are the
hydrogens. A rule of thumb worth remembering: **~110 Da per residue, ~8 heavy atoms per residue**.

### 6.5 Painting atoms into voxels ("splatting")

Heavier atoms scatter electrons more strongly. A simple, common approximation weights each atom by its **atomic number**
Z (its number of electrons: C = 6, N = 7, O = 8, S = 16); real electron-scattering factors are more complicated, but this
is good enough at 10 Å. The recipe for a map at voxel size $a$ (here 10 Å):

1. convert each atom's coordinates to voxel indices: $i = \text{round}\big((x - x_c)/a + (n-1)/2\big)$, where $x_c$ is the
   centre of the molecule and $n$ the box size;
2. add Z to that voxel;
3. blur with a 3D Gaussian of width $\sigma$ (voxels) to mimic finite resolution.

⚠️ **NumPy trap:** with "fancy" (array) indexing, `vol[iz, iy, ix] += w` adds only **once** per voxel even if several atoms
fall into the same voxel. `np.add.at(vol, (iz, iy, ix), w)` accumulates correctly. Let us prove it:

In [ ]:
# ── 6.5 The repeated-index trap ──
ATOMIC_NUMBER = {"H": 1, "C": 6, "N": 7, "O": 8, "P": 15, "S": 16}
Z = np.array([ATOMIC_NUMBER.get(e, 6) for e in atoms["element"]], dtype=np.float32)
n_box, voxel_A = 52, 10.0
centre = xyz.mean(axis=0)
idx = np.round((xyz - centre) / voxel_A + (n_box - 1) / 2).astype(int)    # columns: x, y, z indices

wrong = np.zeros((n_box,) * 3, np.float32)
wrong[idx[:, 2], idx[:, 1], idx[:, 0]] += Z                               # looks right... but is not
right = np.zeros((n_box,) * 3, np.float32)
np.add.at(right, (idx[:, 2], idx[:, 1], idx[:, 0]), Z)                    # accumulates every atom
print(f"total electrons in the model: {Z.sum():,.0f}")
print(f"  with  vol[...] += Z : {wrong.sum():,.0f}   ← most atoms lost!")
print(f"  with  np.add.at     : {right.sum():,.0f}")
print(f"occupied voxels: {(right > 0).sum()} (≈ {Z.size / max((right > 0).sum(), 1):.0f} atoms per 10-Å voxel)")

In [ ]:
# ── 6.6 atoms → density map (the correct recipe as a function), next to PolNet's template of the same entry ──
def atoms_to_density(xyz: np.ndarray, weights: np.ndarray, n: int = 52, voxel_A: float = 10.0,
                     sigma_vox: float = 0.0) -> np.ndarray:
    """Nearest-voxel splat of atoms (weights, e.g. atomic numbers) into an n³ [z, y, x] grid, then Gaussian blur."""
    idx = np.round((xyz - xyz.mean(axis=0)) / voxel_A + (n - 1) / 2).astype(int)
    keep = np.all((idx >= 0) & (idx < n), axis=1)                  # ignore atoms outside the box
    vol = np.zeros((n, n, n), np.float32)
    np.add.at(vol, (idx[keep, 2], idx[keep, 1], idx[keep, 0]), weights[keep])   # x, y, z indices → [z, y, x]
    return ndimage.gaussian_filter(vol, sigma_vox) if sigma_vox > 0 else vol


def center_by_mass(vol: np.ndarray) -> np.ndarray:
    """Shift a map (linear interpolation) so that its centre of mass sits at the centre of the box."""
    com = np.array(ndimage.center_of_mass(np.clip(vol, 0, None)))
    return ndimage.shift(vol, (np.array(vol.shape) - 1) / 2 - com, order=1)


rubisco_ref, ref_voxel, ref_src = fetch_zoo_map("1bxn")
print(f"reference: {ref_src}, shape {rubisco_ref.shape}, {ref_voxel:.0f} Å/voxel")
n_ref = rubisco_ref.shape[0]
ours = center_by_mass(atoms_to_density(xyz, Z, n=n_ref, voxel_A=ref_voxel, sigma_vox=0.75))
ref = center_by_mass(rubisco_ref)

fig, axes = plt.subplots(2, 3, figsize=(8, 5.4))
crop = slice(10, 42)                                            # show the central 32 × 32 nm only
for j, (axis, lab) in enumerate([(0, "down z"), (1, "down y"), (2, "down x")]):
    axes[0, j].imshow(ours.sum(axis=axis)[crop, crop]); axes[0, j].set_title(f"our map from atoms, {lab}", fontsize=8)
    axes[1, j].imshow(ref.sum(axis=axis)[crop, crop]); axes[1, j].set_title(f"PolNet template, {lab}", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle(f"RuBisCO (L$_4$S$_4$) projections — Pearson r = {pearson(ours, ref):.3f}", fontsize=10)
plt.tight_layout(); plt.show()

### 6.7 How similar are the two maps? Pearson correlation

The standard similarity score between two maps $A$ and $B$ with $N$ voxels is the **Pearson correlation coefficient**
(in the fitting literature: the normalised *cross-correlation coefficient*, CCF):

$$r = \frac{\sum_i (A_i - \bar A)(B_i - \bar B)}{\sqrt{\sum_i (A_i - \bar A)^2}\,\sqrt{\sum_i (B_i - \bar B)^2}}$$

*In words:* subtract each map's mean, multiply voxel by voxel, add up, and divide by the maps' overall spreads. $r = 1$
means identical up to brightness and offset, $r = 0$ means unrelated. Because it ignores overall scale, it does not
matter that PolNet weighted atoms differently from us. Let us find the blur that best matches PolNet's map — and see
what happens if we confuse the axis order.

In [ ]:
# ── 6.7 Correlation versus blur, and the axis-order bug ──
sigmas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
scores = [pearson(center_by_mass(atoms_to_density(xyz, Z, n_ref, ref_voxel, s)), ref) for s in sigmas]
best = sigmas[int(np.argmax(scores))]
swapped = np.transpose(ours, (2, 1, 0))              # the bug: treating [z, y, x] as [x, y, z]
r_swapped = pearson(swapped, ref)
for s, sc in zip(sigmas, scores):
    print(f"  σ = {s:4.2f} voxels ({s * ref_voxel:4.1f} Å) → r = {sc:.3f}")
print(f"best blur σ ≈ {best} voxels;  with x and z swapped: r = {r_swapped:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.plot(np.array(sigmas) * ref_voxel, scores, "o-", color="tab:green", label="our map vs PolNet")
ax.axhline(r_swapped, color="tab:red", ls="--", label="axes swapped (bug)")
ax.set_xlabel("Gaussian blur σ (Å)"); ax.set_ylabel("Pearson r")
ax.set_title("Agreement of the atom-derived map with the PolNet template")
ax.legend(fontsize=8); plt.show()
if pdb_source.startswith("PDB") and ref_src.startswith("PolNet"):
    assert max(scores) > 0.95, "the map built from atoms should match PolNet's template closely"

🔍 **What just happened?** From a text file of atoms you built a density map that agrees with an independent reference
map at r ≈ 0.99 or better — once a small blur is added, because PolNet's recipe (atom weights, how atoms are spread
into voxels) differs slightly from ours. Swapping x and z produces a map that is still "RuBisCO-like" but measurably wrong
(lower r): a mirror/transpose error that is easy to miss by eye and is a classic source of wrong-handed maps.

Too much blur also lowers r: **σ is a resolution knob**. (A common convention, used by ChimeraX's `molmap` command,
is σ ≈ 0.225 × resolution; conventions vary between programs.) Here is what RuBisCO looks like as resolution worsens:

In [ ]:
# ── 6.8 One molecule, four resolutions ──
fig, axes = plt.subplots(1, 4, figsize=(9.5, 2.6))
for ax, s in zip(axes, [0.5, 1.0, 2.0, 3.0]):
    m = center_by_mass(atoms_to_density(xyz, Z, n_ref, ref_voxel, s))
    ax.imshow(m[n_ref // 2, crop, crop]); ax.axis("off")
    ax.set_title(f"σ = {s * ref_voxel:.0f} Å (≈ {s * ref_voxel / 0.225:.0f} Å res.)", fontsize=8)
fig.suptitle("Central slice of the atom-derived RuBisCO map as the blur grows", fontsize=9)
plt.tight_layout(); plt.show()

### 📄 Paper connection — fitting atomic structures into density maps

**Paper:** Zhang S, Vasishtan D, Xu M, Topf M, Alber F (2010). *A fast mathematical programming procedure for
simultaneous fitting of assembly components into cryoEM density maps.* Bioinformatics 26(12):i261–i268 (ISMB 2010).
[doi:10.1093/bioinformatics/btq201](https://doi.org/10.1093/bioinformatics/btq201)

- **Question it answers:** a cryo-EM map of a whole assembly is often low resolution (≈ 5–30 Å). If we know the atomic
  structures of its components, *where* does each one sit in the map? Trying every placement of every component at
  once is computationally impossible.
- **What the paper actually does:** it reduces the assembly map and each component to a small set of **feature points**
  (cluster centres from neural-gas vector quantisation of Laplacian-filtered, thresholded maps); matches the point sets
  by an **integer quadratic program** (relaxed and solved with an interior-point method) that fits *all components
  simultaneously*; generates an ensemble of 500 candidate assemblies; **ranks them by the normalised cross-correlation**
  $\mathrm{CCF}=\frac1N\sum_i\frac{(\rho_t(i)-\langle\rho_t\rangle)(\rho_p(i)-\langle\rho_p\rangle)}{\sigma_t\sigma_p}$ between the
  target map and a map simulated from the placed atoms; and refines the best ones with a density-weighted iterative
  closest point method. Test maps were *simulated from PDB entries* (20 Å resolution, 3 Å voxels, with Situs' PDB2VOL).
- **What our toy keeps:** the two ingredients you built in Section 6 — simulating a map from atoms, and the
  correlation score ($r$ above is the same quantity as their CCF).
- **What our toy leaves out:** feature points, the quadratic program, multiple components, refinement. Module 25 fits
  two zoo components into a composite map.
- **Result reported:** on 11 simulated assemblies, the authors report that the top CCF-ranked solution was correct in
  8 of 11 cases (failures involved pseudo-symmetric components), that refinement brought the best solutions to
  0.4–1.8 Å Cα RMSD, and that one IQP run takes 0.10–2.56 s; on the experimental GroEL map EMD-1046 (23.5 Å) the
  lowest RMSD was 8.6 Å.
- **Code:** MATLAB, available from the authors on request (not in AITom).

### 📄 Paper connection (preview) — describing a map with rotation-invariant numbers

**Paper:** Xu M, Zhang S, Alber F (2009). *3D Rotation Invariant Features for the Characterization of Molecular Density
Maps.* IEEE BIBM 2009, pp. 74–78. [doi:10.1109/BIBM.2009.32](https://doi.org/10.1109/BIBM.2009.32)

> ⚠️ We could not access the full text of this paper while writing; the description below is limited to its title and
> abstract. Read the paper before relying on details.

- **Question it answers:** our correlation $r$ only works if the two maps are in the *same orientation*. How can we
  compare density around a point when the molecules are arbitrarily rotated and noisy?
- **What the paper does (abstract level):** it builds and compares three kinds of **rotation-invariant** local
  descriptors of density maps — spherical-harmonics decompositions, moments of the density, and intensity histograms —
  with FFT-based computation, and tests them in a classification of protein density maps at several noise levels.
- **Result reported:** the abstract reports that all three perform similarly at low noise, and that the
  spherical-harmonics features are clearly more reliable at high noise.
- **Where next:** you will implement a spherical-harmonic shell descriptor yourself in
  [Module 02](02_rotations_geometry.ipynb). Code: none in AITom.

---
## 7 · Reproducibility: seeds, saved figures, saved arrays

Computational results must be **reproducible**: you (or a reviewer) should get the same numbers when re-running the
notebook. Three habits:

1. **Seed every random number generator.** A generator created with `np.random.default_rng(seed)` produces the same
   "random" sequence every time for the same seed.
2. **Save figures and arrays** with descriptive names (`fig.savefig`, `np.save`, or `write_mrc` for maps).
3. **Record versions** of the software you used (printed in the setup cell).

In [ ]:
# ── 7.1 Seeds make "random" repeatable ──
a1 = np.random.default_rng(42).normal(size=3)
a2 = np.random.default_rng(42).normal(size=3)
a3 = np.random.default_rng(43).normal(size=3)
print("seed 42:", np.round(a1, 4))
print("seed 42:", np.round(a2, 4), "→ identical:", np.array_equal(a1, a2))
print("seed 43:", np.round(a3, 4), "→ different")

In [ ]:
# ── 7.2 Save a figure, an array and a provenance note ──
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(ribo.sum(axis=0)); ax.set_title("70S ribosome, projection along z"); ax.axis("off")
fig_path = os.path.join(out_dir, "ribosome_projection.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)                                         # saved to disk, not shown again
np.save(os.path.join(out_dir, "rubisco_from_atoms.npy"), ours)
with open(os.path.join(out_dir, "provenance.txt"), "w") as f:
    f.write(f"seed={SEED}\nnumpy={np.__version__}\nscipy={scipy.__version__}\n"
            f"ribosome map: {ribo_src}\nRuBisCO atoms: {pdb_source}\nRuBisCO reference: {ref_src}\n")
print("saved:", sorted(os.listdir(out_dir)))

---
## ✅ Checkpoint

1. A tomogram array has shape `(200, 512, 512)`. How many x-columns does it have, and which index would you change to
   move "up" through the specimen?
   <details><summary>Answer</summary>512 columns along x (the last axis). Moving up through the specimen means changing
   the first index, z (0 … 199).</details>
2. What does `vol.sum(axis=0)` compute, physically?
   <details><summary>Answer</summary>The projection along z — the sum of density along the beam direction, i.e. the
   ideal image an electron microscope records (projection approximation).</details>
3. A map has NX = 100 and a cell size of 354 Å along x. What is its voxel size, and what is the finest periodic detail it
   can represent?
   <details><summary>Answer</summary>354 / 100 = 3.54 Å per voxel. The Nyquist limit is 2 × 3.54 ≈ 7.1 Å.</details>
4. Why does `vol[iz, iy, ix] += w` give the wrong answer when building a map from atoms?
   <details><summary>Answer</summary>With array indices, repeated voxel positions are written only once (the last write
   wins), so atoms that share a voxel are lost. `np.add.at` accumulates all of them.</details>
5. You threshold a map at a very low value and get a huge "molecule". At a very high value it breaks into pieces. How
   could you choose a principled threshold?
   <details><summary>Answer</summary>Choose the level at which the enclosed volume equals the volume expected from the
   mass (≈ M / 0.81 Da/Å³ for protein), or a stated multiple of the noise standard deviation — and always report it.</details>
6. The PDB file 1BXN contains four large and four small RuBisCO subunits. Is that the whole enzyme?
   <details><summary>Answer</summary>No. It is the crystal's asymmetric unit, L₄S₄ — half of the L₈S₈ form I enzyme. The
   full assembly is generated by the symmetry operations in REMARK 350.</details>

## 🧠 Questions & discussion

1. Our map from atoms used atomic number Z as the weight of each atom. Electron microscopes actually measure
   electrostatic potential, X-ray crystallography measures electron density. Why might a map computed from atoms
   disagree with a real cryo-EM map even at the same resolution? (Think: hydrogens, charges, solvent, flexibility.)
2. Binning makes computation faster but discards detail. For *picking* ribosomes in a tomogram, and for *resolving*
   α-helices in an average, which voxel sizes would you choose, and why?
3. The correlation $r$ ignores brightness and offset. Give one situation where that is helpful and one where it could
   mislead you.
4. A collaborator sends a map that looks like the right molecule but the mirror image of the published one. List three
   places in a pipeline (file header, array indexing, display) where such a flip could have happened.
5. Why is reproducibility (seeds, saved versions, saved parameters) especially important for methods that average
   thousands of noisy particles?

## 🚀 Bonus — going further

- **More molecules:** build maps from other PDB files in PolNet's `data/templates/pdbs/` folder (same pinned commit),
  e.g. `3j9i.pdb` (20S proteasome), and compare with the corresponding `mrcs_10A` templates.
- **The full RuBisCO:** read the `REMARK 350 BIOMT` lines of 1BXN, apply the second operation (a 3×3 rotation plus a
  shift) to every atom, and build the complete L₈S₈ map. (Rotations are the topic of Module 02.)
- **Better splatting:** spread each atom over its 8 neighbouring voxels with trilinear weights instead of the nearest
  voxel, and see whether the agreement with PolNet improves at σ = 0.
- **Real maps:** download a map from the [EMDB](https://www.ebi.ac.uk/emdb/) (e.g. EMD-2682, a mammalian ribosome
  complex) and open it with `read_mrc`; check its header words.
- **Deep dive on tensors:** [PyTorch tensor basics](../../../../sci_img/tutorial/ComputationalSignalBase/PyTorchTensorBasic/) —
  PyTorch tensors behave like NumPy arrays and are used from Module 05 on.

## 📝 Lab note

```text
Module 01 — date:
Input: PolNet maps (4V4R, 3J9I, 1BXN), NGL betaGal.mrc (3.54 Å/voxel), PDB 1BXN atoms
Output: slices, projections, isosurfaces, binned maps, my own MRC file, a map built from atoms
Method: array indexing, sum(axis), thresholds, block-average binning, struct header decoding, np.add.at splatting,
        Gaussian blur, Pearson correlation
Key parameter changed & effect (e.g. threshold, bin factor, blur σ):
Failure mode I hit (e.g. view vs copy, axis order, repeated indices):
Paper link: Zhang et al. 2010 (CCF ranking of fits); Xu, Zhang & Alber 2009 (preview)
```

## 📚 References

1. Zhang S, Vasishtan D, Xu M, Topf M, Alber F. A fast mathematical programming procedure for simultaneous fitting of
   assembly components into cryoEM density maps. *Bioinformatics* 26(12):i261–i268 (2010). doi:10.1093/bioinformatics/btq201
2. Xu M, Zhang S, Alber F. 3D rotation invariant features for the characterization of molecular density maps.
   *IEEE BIBM 2009*, pp. 74–78. doi:10.1109/BIBM.2009.32
3. Cheng A, Henderson R, Mastronarde D, *et al.* MRC2014: extensions to the MRC format header for electron cryo-microscopy
   and tomography. *J Struct Biol* 192(2):146–150 (2015). doi:10.1016/j.jsb.2015.04.002
4. Hansen S, Vollan VB, Hough E, Andersen K. The crystal structure of rubisco from *Alcaligenes eutrophus* reveals a novel
   central eight-stranded β-barrel formed by β-strands from four subunits. *J Mol Biol* 288:609 (1999).
   doi:10.1006/jmbi.1999.2701 — PDB 1BXN (2.7 Å X-ray structure; *Alcaligenes eutrophus* is an older name of *C. necator*).
5. Harris CR *et al.* Array programming with NumPy. *Nature* 585:357–362 (2020). doi:10.1038/s41586-020-2649-2
6. Martinez-Sanchez A *et al.* PolNet — simulator of cryo-ET tomograms (source of the density templates and PDB files;
   Apache-2.0). <https://github.com/anmartinezs/polnet>
7. NGL Viewer test data (β-galactosidase map, MIT licence). <https://github.com/nglviewer/ngl>

## ➡️ Next

**[02 · Geometry of molecules: coordinates, rotations, symmetry and invariance](02_rotations_geometry.ipynb)** — every
particle in a cell points in a random direction. Next you will learn to rotate maps and atoms, measure how different two
orientations are, find a molecule's symmetry axis, and build features that do not change when a molecule turns.
Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).